# WarpKriging with no warping (baseline) (Python)

This notebook demonstrates `WarpKriging` with `warping='none'` — an identity warp that
behaves like standard Kriging. It serves as a baseline for comparing other warp methods.

Steps:
1. Install pylibkriging (run once)
2. Load pylibkriging
3. Define the Branin function and plot it
4. Build a space-filling design and evaluate it
5. Fit a `WarpKriging` model
6. Predict on a fine grid and plot mean + uncertainty
7. Inspect model parameters

## 0. Installation (run once)

Build and install **pylibkriging** from the local source tree.
Requires: `cmake`, a C++ compiler, Python ≥ 3.7.

In [ ]:
import sys, os
repo_root = os.path.normpath(os.path.join(os.getcwd(), '..', '..'))
venv_bin  = os.path.join(repo_root, 'bindings', 'Python', 'venv', 'bin')

if not os.path.exists(os.path.join(venv_bin, 'python')):
    !{sys.executable} -m venv {repo_root}/bindings/Python/venv

!{venv_bin}/pip install -q -r {repo_root}/bindings/Python/pylibkriging/requirements.txt \
                            -r {repo_root}/bindings/Python/pylibkriging/dev-requirements.txt

!cd {repo_root} && \
    EXTRA_CMAKE_OPTIONS="-DPYTHON_EXECUTABLE={venv_bin}/python" \
    ENABLE_PYTHON_BINDING=on \
    bash tools/linux-macos/build.sh

In [ ]:
# Install the Python package into the current kernel
%pip install --no-build-isolation ../../bindings/Python/pylibkriging/

## 1. Load pylibkriging

In [ ]:
import numpy as np
!{venv_bin}/pip install -q matplotlib
import matplotlib.pyplot as plt
import pylibkriging as lk

print("pylibkriging version:", lk.__version__)

## 2. Branin function

The Branin function is a standard benchmark for surrogate modelling, defined on $[0,1]^2$
(rescaled from its canonical domain $[-5, 10] \times [0, 15]$).

In [ ]:
def branin(X):
    X = np.atleast_2d(X)
    x1 = X[:, 0] * 15 - 5
    x2 = X[:, 1] * 15
    return (
        (x2 - 5 / (4 * np.pi**2) * x1**2 + 5 / np.pi * x1 - 6) ** 2
        + 10 * (1 - 1 / (8 * np.pi)) * np.cos(x1)
        + 10
    )

# 50x50 evaluation grid
grid_x = np.linspace(0, 1, 50)
G1, G2  = np.meshgrid(grid_x, grid_x)
grid    = np.column_stack([G1.ravel(), G2.ravel()])
z_true  = branin(grid).reshape(50, 50)

plt.figure(figsize=(5, 4))
plt.contourf(G1, G2, z_true, levels=20, cmap='terrain')
plt.colorbar(label='Branin(x)')
plt.title('True Branin function')
plt.xlabel('$x_1$'); plt.ylabel('$x_2$')
plt.tight_layout()
plt.show()

## 3. Design of experiments

We sample $n = 30$ points using a Latin Hypercube Design.

In [ ]:
rng = np.random.default_rng(42)

def lhs(n, d, rng):
    """Simple LHS: stratified uniform sample, independently permuted per dimension."""
    X = np.empty((n, d))
    for j in range(d):
        perm = rng.permutation(n)
        X[:, j] = (perm + rng.uniform(size=n)) / n
    return X

n = 30
X = lhs(n, 2, rng)
y = branin(X)

plt.figure(figsize=(5, 4))
plt.contourf(G1, G2, z_true, levels=20, cmap='terrain', alpha=0.7)
plt.scatter(X[:, 0], X[:, 1], c='white', edgecolors='black', s=60, zorder=5)
plt.colorbar(label='Branin(x)')
plt.title(f'{n} LHS design points on Branin')
plt.xlabel('$x_1$'); plt.ylabel('$x_2$')
plt.tight_layout()
plt.show()

## 4. Fit a WarpKriging model (`none`)

We use `warping=['none', 'none']` — one warp per input dimension.

In [ ]:
wk = lk.WarpKriging(
    y, X,
    warping=['none', 'none'],
    kernel='matern5_2',
    optim='BFGS',
    parameters={}
)
print(wk.summary())

## 5. Predict on a fine grid

`predict()` returns `(mean, stdev, cov)`.

In [ ]:
pred   = wk.predict(grid)
z_mean = pred[0].reshape(50, 50)
z_sd   = pred[1].reshape(50, 50)

vmin = min(z_true.min(), z_mean.min())
vmax = max(z_true.max(), z_mean.max())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, z, title in zip(axes, [z_true, z_mean], ['True Branin', f'WarpKriging ({warp_name}) mean']):
    cf = ax.contourf(G1, G2, z, levels=20, cmap='terrain', vmin=vmin, vmax=vmax)
    ax.contour(G1, G2, z, levels=10, colors='grey', linewidths=0.5)
    ax.scatter(X[:, 0], X[:, 1], c='white', edgecolors='black', s=40, zorder=5)
    ax.set_title(title)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    fig.colorbar(cf, ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
# Posterior standard deviation (uncertainty)
plt.figure(figsize=(5, 4))
cf = plt.contourf(G1, G2, z_sd, levels=20, cmap='YlOrRd_r')
plt.scatter(X[:, 0], X[:, 1], c='white', edgecolors='black', s=60, zorder=5)
plt.colorbar(cf, label='Std dev')
plt.title('WarpKriging (none) std dev (uncertainty)')
plt.xlabel('$x_1$'); plt.ylabel('$x_2$')
plt.tight_layout()
plt.show()

## 6. Model inspection

Key fitted parameters: length-scales $\theta$, variance $\sigma^2$, log-likelihood, and warping specification.

In [ ]:
print(f"Kernel       : {wk.kernel()}")
print(f"Theta (range): {np.round(wk.theta(), 4)}")
print(f"Sigma2       : {wk.sigma2():.4f}")
print(f"LogLikelihood: {wk.logLikelihood():.4f}")
print(f"Feature dim  : {wk.feature_dim()}")
print(f"Warping      : {wk.warping()}")